# Leather RePaint Dataset Preparation

This notebook converts the Leather dataset stored under `/work/Ayan/Leather` into a RePaint-style training dataset under `/work/Ayan/repaint` and provides a starter training loop you can extend or swap with a more advanced diffusion-based pipeline.

**Workflow**
- Inspect and configure source/target paths.
- Convert `control_1`, `control_2`, `images`, and prompts into a unified dataset with metadata.
- (Optional) Sanity-check generated samples.
- Launch a lightweight training loop that learns to reconstruct the target image given a source image and mask.

> ⚠️ Update the paths or hyperparameters as needed before running on your environment.

In [ ]:
%pip install --quiet --upgrade pillow tqdm torch torchvision matplotlib

In [ ]:
from pathlib import Path
import shutil
import json
from typing import Dict, List, Optional, Tuple

import numpy as np
from PIL import Image
from tqdm.auto import tqdm

SOURCE_ROOT = Path("/work/Ayan/Leather")
OUTPUT_ROOT = Path("/work/Ayan/repaint")
SPLITS = ("train", "test")

if not SOURCE_ROOT.exists():
    raise FileNotFoundError(f"Source dataset not found at {SOURCE_ROOT}. Update SOURCE_ROOT and retry.")

print(f"Source root: {SOURCE_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")

In [ ]:
def ensure_clean_dir(path: Path, recreate: bool = True) -> None:
    """Create a clean directory when `recreate=True`, else ensure it exists."""
    if path.exists() and recreate:
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def load_prompts(split_dir: Path) -> Tuple[Dict[str, str], Optional[str]]:
    """Load prompts for a split.

    Supports either a single prompt in `prompt.txt`, per-sample mappings in the same file
    delimited by `,` or `|`, or individual txt files in a `prompts/` directory.
    Returns `(prompt_map, default_prompt)`.
    """
    prompt_map: Dict[str, str] = {}
    default_prompt: Optional[str] = None

    prompt_file = split_dir / "prompt.txt"
    if prompt_file.exists():
        lines = [line.strip() for line in prompt_file.read_text(encoding="utf-8").splitlines() if line.strip()]
        structured_lines = [line for line in lines if ("|" in line) or ("," in line)]
        if structured_lines:
            for line in structured_lines:
                if "|" in line:
                    stem, prompt = line.split("|", 1)
                else:
                    stem, prompt = line.split(",", 1)
                prompt_map[stem.strip()] = prompt.strip()
        else:
            default_prompt = " ".join(lines)
    else:
        prompt_dir = split_dir / "prompts"
        if prompt_dir.exists():
            for txt_file in prompt_dir.glob("*.txt"):
                prompt_map[txt_file.stem] = txt_file.read_text(encoding="utf-8").strip()

    return prompt_map, default_prompt


def build_binary_mask(mask_img: Image.Image) -> Image.Image:
    """Convert an RGB mask to binary (0 or 255) mask."""
    rgb = np.array(mask_img.convert("RGB"))
    binary = (rgb.sum(axis=-1) > 0).astype(np.uint8) * 255
    return Image.fromarray(binary, mode="L")


def collect_split_samples(split_dir: Path) -> List[str]:
    images_dir = split_dir / "images"
    if not images_dir.exists():
        raise FileNotFoundError(f"Expected 'images' directory under {split_dir}")
    return sorted({path.stem for path in images_dir.glob("*.*")})

In [ ]:
def convert_split(split: str, recreate_output: bool = False) -> Dict[str, int]:
    split_dir = SOURCE_ROOT / split
    if not split_dir.exists():
        raise FileNotFoundError(f"Missing split directory: {split_dir}")

    prompt_map, default_prompt = load_prompts(split_dir)
    samples = collect_split_samples(split_dir)

    out_split_dir = OUTPUT_ROOT / split
    (out_split_dir / "target").mkdir(parents=True, exist_ok=True)
    (out_split_dir / "source").mkdir(parents=True, exist_ok=True)
    (out_split_dir / "mask").mkdir(parents=True, exist_ok=True)
    (out_split_dir / "mask_multiclass").mkdir(parents=True, exist_ok=True)

    metadata: List[str] = []
    skipped = 0

    for stem in tqdm(samples, desc=f"{split} conversion"):
        target_path = split_dir / "images" / f"{stem}.png"
        if not target_path.exists():
            target_candidates = list((split_dir / "images").glob(f"{stem}.*"))
            if not target_candidates:
                skipped += 1
                continue
            target_path = target_candidates[0]

        source_path = split_dir / "control_2" / target_path.name
        mask_path = split_dir / "control_1" / target_path.name

        if not source_path.exists() or not mask_path.exists():
            skipped += 1
            continue

        target_img = Image.open(target_path).convert("RGB")
        source_img = Image.open(source_path).convert("RGB")
        mask_img = Image.open(mask_path)

        binary_mask = build_binary_mask(mask_img)

        out_target = out_split_dir / "target" / target_path.name
        out_source = out_split_dir / "source" / target_path.name
        out_mask = out_split_dir / "mask" / target_path.name
        out_mask_mc = out_split_dir / "mask_multiclass" / target_path.name

        target_img.save(out_target)
        source_img.save(out_source)
        binary_mask.save(out_mask)
        mask_img.save(out_mask_mc)

        prompt = prompt_map.get(stem, default_prompt or "")
        metadata.append(json.dumps({
            "id": stem,
            "target": f"target/{target_path.name}",
            "source": f"source/{target_path.name}",
            "mask": f"mask/{target_path.name}",
            "mask_multiclass": f"mask_multiclass/{target_path.name}",
            "prompt": prompt
        }))

    metadata_path = out_split_dir / "metadata.jsonl"
    metadata_path.write_text("\n".join(metadata), encoding="utf-8")

    return {
        "total_samples": len(samples),
        "converted": len(metadata),
        "skipped": skipped,
        "metadata_file": str(metadata_path)
    }


def convert_dataset(recreate_output: bool = True) -> Dict[str, Dict[str, int]]:
    if recreate_output:
        ensure_clean_dir(OUTPUT_ROOT, recreate=True)
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    stats = {}
    for split in SPLITS:
        stats[split] = convert_split(split, recreate_output=False)
    return stats

In [ ]:
conversion_stats = convert_dataset(recreate_output=True)
conversion_stats

In [ ]:
import matplotlib.pyplot as plt

example_split = "train"
metadata_path = OUTPUT_ROOT / example_split / "metadata.jsonl"
if metadata_path.exists():
    sample_meta = json.loads(metadata_path.read_text(encoding="utf-8").splitlines()[0])
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, key in zip(axes, ["source", "mask", "mask_multiclass", "target"]):
        img = Image.open(OUTPUT_ROOT / example_split / sample_meta[key]).convert("RGB")
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(key)
    plt.suptitle(f"Sample: {sample_meta['id']} | Prompt: {sample_meta['prompt']}")
else:
    print(f"Metadata not found at {metadata_path}")

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import functional as F


class LeatherRepaintDataset(Dataset):
    def __init__(self, metadata_file: Path, root_dir: Path, target_size: Tuple[int, int] = (512, 512), augment: bool = False):
        if not metadata_file.exists():
            raise FileNotFoundError(f"Metadata file not found: {metadata_file}")
        self.entries = [json.loads(line) for line in metadata_file.read_text(encoding="utf-8").splitlines() if line.strip()]
        self.root_dir = root_dir
        self.augment = augment
        self.target_size = target_size

        self.image_transform = transforms.Compose([
            transforms.Resize(target_size, interpolation=Image.BICUBIC),
            transforms.ToTensor()
        ])

    def __len__(self) -> int:
        return len(self.entries)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        entry = self.entries[idx]
        source = Image.open(self.root_dir / entry["source"]).convert("RGB")
        target = Image.open(self.root_dir / entry["target"]).convert("RGB")
        mask = Image.open(self.root_dir / entry["mask"]).convert("L")

        source_tensor = self.image_transform(source)
        target_tensor = self.image_transform(target)
        mask = mask.resize(self.target_size, Image.NEAREST)
        mask_tensor = F.to_tensor(mask)
        mask_tensor = (mask_tensor > 0.5).float()

        if self.augment:
            if torch.rand(1).item() > 0.5:
                source_tensor = torch.flip(source_tensor, dims=[2])
                target_tensor = torch.flip(target_tensor, dims=[2])
                mask_tensor = torch.flip(mask_tensor, dims=[2])
            if torch.rand(1).item() > 0.5:
                source_tensor = torch.flip(source_tensor, dims=[1])
                target_tensor = torch.flip(target_tensor, dims=[1])
                mask_tensor = torch.flip(mask_tensor, dims=[1])

        model_input = torch.cat([source_tensor, mask_tensor], dim=0)

        return {
            "id": entry["id"],
            "prompt": entry.get("prompt", ""),
            "input": model_input,
            "target": target_tensor,
            "mask": mask_tensor
        }


class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, mid_channels: Optional[int] = None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.double_conv(x)


class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.maxpool_conv(x)


class Up(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels // 2, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        x1 = self.up(x1)
        diff_y = x2.size()[2] - x1.size()[2]
        diff_x = x2.size()[3] - x1.size()[3]

        x1 = nn.functional.pad(x1, [diff_x // 2, diff_x - diff_x // 2,
                                    diff_y // 2, diff_y - diff_y // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(x)


class SimpleUNet(nn.Module):
    def __init__(self, in_channels: int = 4, out_channels: int = 3, bilinear: bool = True, base_features: int = 32):
        super().__init__()
        factor = 2 if bilinear else 1
        self.inc = DoubleConv(in_channels, base_features)
        self.down1 = Down(base_features, base_features * 2)
        self.down2 = Down(base_features * 2, base_features * 4)
        self.down3 = Down(base_features * 4, base_features * 8)
        self.down4 = Down(base_features * 8, base_features * 16 // factor)
        self.up1 = Up(base_features * 16, base_features * 8 // factor, bilinear)
        self.up2 = Up(base_features * 8, base_features * 4 // factor, bilinear)
        self.up3 = Up(base_features * 4, base_features * 2 // factor, bilinear)
        self.up4 = Up(base_features * 2, base_features, bilinear)
        self.outc = OutConv(base_features, out_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        output = self.outc(x)
        return torch.sigmoid(output)

In [ ]:
from datetime import datetime


def train_model(
    train_metadata: Path,
    val_metadata: Optional[Path],
    epochs: int = 10,
    batch_size: int = 2,
    learning_rate: float = 1e-4,
    target_size: Tuple[int, int] = (512, 512),
    checkpoint_dir: Optional[Path] = None,
    num_workers: int = 4,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_dataset = LeatherRepaintDataset(train_metadata, train_metadata.parent, target_size=target_size, augment=True)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)

    val_loader = None
    if val_metadata and val_metadata.exists():
        val_dataset = LeatherRepaintDataset(val_metadata, val_metadata.parent, target_size=target_size, augment=False)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

    model = SimpleUNet(in_channels=4, out_channels=3).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    def reconstruction_loss(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        inside = torch.abs(pred - target) * mask
        outside = torch.abs(pred - target) * (1 - mask)
        loss_inside = inside.view(inside.size(0), -1).mean(dim=1)
        loss_outside = outside.view(outside.size(0), -1).mean(dim=1)
        return (loss_inside + 0.1 * loss_outside).mean()

    best_val_loss = float("inf")
    if checkpoint_dir is None:
        checkpoint_dir = OUTPUT_ROOT / "checkpoints"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}"):
            inputs = batch["input"].to(device)
            targets = batch["target"].to(device)
            masks = batch["mask"].to(device)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(inputs)
                loss = reconstruction_loss(outputs, targets, masks)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * inputs.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        log_message = f"Epoch {epoch}: train_loss={train_loss:.5f}"

        if val_loader is not None:
            model.eval()
            val_loss_accum = 0.0
            with torch.no_grad():
                for batch in val_loader:
                    inputs = batch["input"].to(device)
                    targets = batch["target"].to(device)
                    masks = batch["mask"].to(device)
                    outputs = model(inputs)
                    loss = reconstruction_loss(outputs, targets, masks)
                    val_loss_accum += loss.item() * inputs.size(0)
            val_loss = val_loss_accum / len(val_loader.dataset)
            log_message += f", val_loss={val_loss:.5f}"

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_path = checkpoint_dir / "best_model.pt"
                torch.save({
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_loss": val_loss
                }, best_path)
        else:
            val_loss = None

        print(log_message)

    final_path = checkpoint_dir / f"model_final_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pt"
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss if val_loader is not None else None,
        "config": {
            "epochs": epochs,
            "batch_size": batch_size,
            "learning_rate": learning_rate,
            "target_size": target_size,
        }
    }, final_path)

    print(f"Training complete. Final checkpoint saved to {final_path}")
    return model


train_metadata_path = OUTPUT_ROOT / "train" / "metadata.jsonl"
val_metadata_path = OUTPUT_ROOT / "test" / "metadata.jsonl"

# Adjust hyperparameters as needed before running.
model = train_model(
    train_metadata=train_metadata_path,
    val_metadata=val_metadata_path if val_metadata_path.exists() else None,
    epochs=10,
    batch_size=4,
    learning_rate=5e-4,
    target_size=(512, 512),
    num_workers=4,
)